# Malaria Cell Classification: Custom CNN vs MobileNetV2

This notebook compares two Deep Learning architectures on the **TFDS Malaria** dataset (27,558 thin blood smear cell images, binary classification: `Parasitized` vs `Uninfected`):
1. **Custom CNN (Baseline):** 3-block `Conv2D` + `BatchNormalization` + `MaxPooling2D` (`32 -> 64 -> 128`) with `GlobalAveragePooling2D`, `Dropout(0.4)`, and `Dense(128)`.
2. **MobileNetV2 (Transfer Learning):** ImageNet-pretrained `MobileNetV2` with Phase 1 (frozen feature extraction) and Phase 2 (fine-tuning top 30 layers at `lr=1e-5`).

In [ ]:
import os
import sys
import subprocess
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Auto-clone repository when opened directly in Google Colab
if "google.colab" in sys.modules and not os.path.exists("src"):
    repo_dir = "Malaria-Cell-Classification"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "https://github.com/twelverakki/Malaria-Cell-Classification.git"], check=True)
    os.chdir(repo_dir)

# Ensure project root is on sys.path (works from notebooks/ or repo root / Colab)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices("GPU"))

## 1. Dataset Loading & Preprocessing (`tf.data`)

In [ ]:
from src.data import load_malaria_datasets

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds, val_ds, test_ds, class_names, split_counts = load_malaria_datasets(
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

print("Label mapping:", {idx: name for idx, name in enumerate(class_names)})
print("Split counts:", split_counts)

In [ ]:
# Preview sample training images
sample_images, sample_labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.ravel()):
    ax.imshow(np.clip(sample_images[idx].numpy(), 0, 255).astype(np.uint8))
    ax.set_title(class_names[int(sample_labels[idx])])
    ax.axis("off")
plt.suptitle("Sample Augmented Training Images", fontsize=13)
plt.tight_layout()
plt.show()

## 2. Model 1: Custom CNN (Baseline)

In [ ]:
from src.models import build_custom_cnn, get_callbacks

cnn_model = build_custom_cnn(input_shape=(128, 128, 3), dropout_rate=0.4, learning_rate=1e-3)
cnn_model.summary()

cnn_history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=get_callbacks(patience=3),
)

In [ ]:
from src.evaluate import (
    evaluate_model,
    plot_training_history,
    plot_confusion_matrix,
    plot_sample_predictions,
)

plot_training_history(
    cnn_history.history,
    model_name="Custom CNN",
    save_path=os.path.join(OUTPUT_DIR, "custom_cnn_history.png"),
)

cnn_results = evaluate_model(cnn_model, test_ds, model_name="Custom CNN")
print("Custom CNN Test Metrics:", cnn_results["metrics"])

plot_confusion_matrix(
    cnn_results["y_true"],
    cnn_results["y_pred"],
    class_names=class_names,
    model_name="Custom CNN",
    save_path=os.path.join(OUTPUT_DIR, "custom_cnn_cm.png"),
)

plot_sample_predictions(
    cnn_model,
    test_ds,
    class_names=class_names,
    model_name="Custom CNN",
    save_path=os.path.join(OUTPUT_DIR, "custom_cnn_samples.png"),
)

## 3. Model 2: Transfer Learning (`MobileNetV2`)

- **Phase 1:** Feature extraction with frozen ImageNet backbone (10 epochs, `lr=1e-3`).
- **Phase 2:** Fine-tuning top 30 layers (up to 10 epochs, `lr=1e-5`).

In [ ]:
from src.models import build_mobilenetv2, unfreeze_for_finetuning

mobilenet_model, base_model = build_mobilenetv2(
    input_shape=(128, 128, 3),
    dropout_rate=0.3,
    learning_rate=1e-3,
)
mobilenet_model.summary()

# Phase 1: Feature Extraction
mn_hist_phase1 = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=get_callbacks(patience=3),
)

In [ ]:
# Phase 2: Fine-Tuning top 30 layers
mobilenet_model = unfreeze_for_finetuning(
    mobilenet_model,
    base_model,
    unfreeze_top_n=30,
    learning_rate=1e-5,
)

mn_hist_phase2 = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=get_callbacks(patience=3),
)

In [ ]:
from src.evaluate import merge_histories

mn_merged_hist, split_epoch = merge_histories(mn_hist_phase1, mn_hist_phase2)

plot_training_history(
    mn_merged_hist,
    model_name="MobileNetV2 (Transfer Learning + Fine-Tuning)",
    save_path=os.path.join(OUTPUT_DIR, "mobilenetv2_history.png"),
    fine_tune_epoch=split_epoch,
)

mn_results = evaluate_model(mobilenet_model, test_ds, model_name="MobileNetV2")
print("MobileNetV2 Test Metrics:", mn_results["metrics"])

plot_confusion_matrix(
    mn_results["y_true"],
    mn_results["y_pred"],
    class_names=class_names,
    model_name="MobileNetV2",
    save_path=os.path.join(OUTPUT_DIR, "mobilenetv2_cm.png"),
)

plot_sample_predictions(
    mobilenet_model,
    test_ds,
    class_names=class_names,
    model_name="MobileNetV2",
    save_path=os.path.join(OUTPUT_DIR, "mobilenetv2_samples.png"),
)

## 4. Side-by-Side Model Comparison

In [ ]:
from src.evaluate import build_comparison_table

comparison_df = build_comparison_table([cnn_results, mn_results], save_dir=OUTPUT_DIR)
display(comparison_df.round(4))